# 16 — Cruscotto Finale per la Tesi: Framework FCGR Ottimizzato vs Baseline & Stato dell'Arte (SOTA)

## Scopo del Notebook (Guida per il Lettore / Relatore)
Questo notebook raccoglie in forma pulita, ordinata e pronta per la relazione di tesi **tutti i risultati ufficiali** prodotti dall'intera pipeline (`01` $\rightarrow$ `09`):
1. **Architettura del Nostro Framework FCGR (4 Varianti)** e guida rapida su quando usare ciascuna variante.
2. **Benchmark Computazionale & Stato dell'Arte (SOTA)**: confronto diretto di **Tempi (`ms/seq`, Speedup)**, **Memoria (`KB/GB`, Risparmio %)** e **Sparsità Spaziale (%)** contro la Baseline originale e le implementazioni SOTA (`complexCGR`, `DictCounter`, `PFCGR`).
3. **Validazione Downstream su Rete Siamese (`SiameseV3`)**: tabella completa di tutti gli esperimenti di classificazione binaria (**Hotspot vs Stabile**, `04`, `06`, `06b`, `07`, `08`) con grafici di **Trade-off Efficienza vs Accuratezza (Frontiera di Pareto)** e esportazione automatica della tabella Markdown per la tesi.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_fcgr_siamese')
if not COLAB_ROOT.exists():
    COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_project')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
COMPARISON_DIR = ARTIFACTS_DIR / 'final_fcgr_comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print('PROJECT_ROOT :', PROJECT_ROOT)

## 1. Tassonomia del Nostro Framework FCGR (4 Varianti Complementari)
| Variante | Obiettivo Principale | Canali / Tipo | Memoria ($k=6$) | Quando usarla |
| :--- | :--- | :---: | :---: | :--- |
| **Var. 0 — `Bitwise + uint8`** | Massima velocità (`~85x–326x`) e minima RAM (`-75%`), **equivalenza matematica esatta** (`err=0`) | `1 ch uint8` | `4 KB/seq` | Sostituzione immediata *drop-in* della FCGR classica su dataset enormi. |
| **Var. 1 — `Opt 1 (RC + Log)`** | Alta efficienza (`-75%` RAM) + invarianza al filamento DNA + compressione outlier | `1 ch uint8` | `4 KB/seq` | Miglior compromesso **Pareto-ottimale** tra RAM minima e `cls_roc_auc` / `gap`. |
| **Var. 2 — `Opt 2 (Multi-Scale)`** | Abbattimento drastico della **sparsità spaziale** (`87% -> 13%`) via piramide frattale $k=4,5,6$ | `3 ch uint8` | `12 KB/seq` | Sequenze corte/sparse dove i filtri convoluzionali necessitano di contesto denso. |
| **Var. 3 — `Opt 3b (O/E + MS)`** | Massima ricchezza biologica: `Log-Prob` + **Arricchimento `O/E`** (rimuove rumore GC) + Multi-Scala bilanciata (`sum=1`) | `3 ch uint8` | `12 KB/seq` | Ricerca di motivi biologici fini sovra-rappresentati (`PR-AUC`, `F1`, `d'`). |

## 2. Benchmark Computazionale: Baseline vs Stato dell'Arte (SOTA) vs Nostro Framework
Carichiamo i risultati ufficiali prodotti dal Notebook `09` (`sota_length_scaling_benchmark.csv`) e `05` (`optimized_fcgr_k_benchmark.csv`).

In [ ]:
sota_path = RESULTS_DIR / 'sota_length_scaling_benchmark.csv'
k_bench_path = RESULTS_DIR / 'optimized_fcgr_k_benchmark.csv'

if sota_path.exists():
    sota_df = pd.read_csv(sota_path)
    N_ECC = 3_755_079
    sub_ecc6 = sota_df[(sota_df['k'] == 6) & (sota_df['L_bp'] == 500)].copy()
    base_t = float(sub_ecc6.loc[sub_ecc6['method'] == '1_Baseline_Python_f32', 'time_ms'].iloc[0])
    base_m = float(sub_ecc6.loc[sub_ecc6['method'] == '1_Baseline_Python_f32', 'mem_KB'].iloc[0])
    sub_ecc6['speedup_vs_base_x'] = (base_t / sub_ecc6['time_ms']).round(1)
    sub_ecc6['ram_saving_pct'] = ((1.0 - sub_ecc6['mem_KB'] / base_m) * 100.0).round(1)
    sub_ecc6['3.75M_eccDNA_min'] = ((sub_ecc6['time_ms'] * N_ECC) / 60_000.0).round(1)
    sub_ecc6['3.75M_eccDNA_GB'] = ((sub_ecc6['mem_KB'] * N_ECC) / (1024 ** 2)).round(2)
    print('=== CONFRONTO SOTA SU INTERO DATASET eccDNA (3.75M seq @ 500 bp, k=6) ===')
    cols_sota = ['method', 'time_ms', 'speedup_vs_base_x', '3.75M_eccDNA_min', 'mem_KB', '3.75M_eccDNA_GB', 'ram_saving_pct', 'spatial_sparsity_pct']
    display(sub_ecc6[[c for c in cols_sota if c in sub_ecc6.columns]])
else:
    print('Esegui prima il Notebook 09 per includere il benchmark SOTA completo!')

if k_bench_path.exists():
    print('\n=== SCALABILITA\' PER ORDINE k (4..10): BASELINE vs BITWISE + UINT8 ===')
    display(pd.read_csv(k_bench_path))

## 3. Validazione Downstream Completa sulla Rete Siamese (`SiameseV3` — Tutti gli Esperimenti)
Raccogliamo automaticamente i risultati di tutti gli esperimenti eseguiti nei Notebook `04`, `06`, `06b`, `07` e `08`, produciamo la **Frontiera di Pareto (RAM vs Performance)** e salviamo la tabella finale pronta per la tesi.

In [ ]:
metric_files = sorted(ARTIFACTS_DIR.glob('siamese_*/*_final_metrics.csv'))
if len(metric_files) > 0:
    dfs = []
    for f in metric_files:
        df_m = pd.read_csv(f)
        df_m.insert(0, 'experiment', f.parent.name)
        if f.parent.name == 'siamese_baseline_k6':
            df_m['in_channels'] = 1
            df_m['cache_mb'] = 937.50
        dfs.append(df_m)
    comp_df = pd.concat(dfs, ignore_index=True).sort_values('cls_roc_auc', ascending=False).reset_index(drop=True)
    comp_df.to_csv(COMPARISON_DIR / 'siamese_binary_comparison_table.csv', index=False)

    cols_focus = [
        'experiment', 'k', 'in_channels', 'cache_mb', 'cls_roc_auc', 'cls_pr_auc', 'cls_f1', 'cls_balanced_accuracy',
        'pairwise_roc_auc', 'd_pos_mean', 'd_neg_mean', 'distance_gap', 'd_prime', 'best_epoch'
    ]
    clean_table = comp_df[[c for c in cols_focus if c in comp_df.columns]].copy()
    print('=== TABELLA FINALE TUTTI GLI ESPERIMENTI SIAMESEV3 (ORDINATI PER cls_roc_auc) ===')
    display(clean_table.round(4))

    # Esportazione pronta in Markdown per la relazione di tesi
    md_path = COMPARISON_DIR / 'thesis_summary_table.md'
    with open(md_path, 'w', encoding='utf-8') as f_md:
        f_md.write(clean_table.round(4).to_markdown(index=False))
    print(f'Tabella Markdown salvata per la tesi in: {md_path}')

    # Grafico Finale a 3 pannelli (Frontiera di Pareto RAM vs ROC-AUC, Distance Gap & d-prime, PR-AUC & F1)
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

    # Pannello 1: Pareto RAM (MB) vs cls_roc_auc
    for _, row in clean_table.iterrows():
        is_base = 'baseline' in row['experiment']
        color = '#C53030' if is_base else ('#2B6CB0' if 'cosine' in row['experiment'] else '#2F855A')
        marker = 'X' if is_base else 'o'
        axes[0].scatter(row['cache_mb'], row['cls_roc_auc'], s=120, color=color, marker=marker, alpha=0.9)
        short_lbl = row['experiment'].replace('siamese_', '')
        axes[0].annotate(short_lbl, (row['cache_mb'], row['cls_roc_auc']), textcoords='offset points', xytext=(6, 4), fontsize=7.5)
    axes[0].set_xlabel('Dimensione Cache 60k Sequenze (MB) — Meno è Meglio')
    axes[0].set_ylabel('Classificazione Binaria ROC-AUC — Più è Meglio')
    axes[0].set_title('Frontiera di Pareto: Efficienza RAM vs ROC-AUC')
    axes[0].grid(alpha=0.3)

    # Pannello 2: Confronto Distance Gap e d-prime
    y_pos = np.arange(len(clean_table))
    short_names = clean_table['experiment'].str.replace('siamese_', '', regex=False).tolist()
    axes[1].barh(y_pos - 0.18, clean_table['distance_gap'], height=0.35, label='Distance Gap (d- - d+)', color='#805AD5')
    axes[1].barh(y_pos + 0.18, clean_table['d_prime'] / 4.0, height=0.35, label="d' / 4 (scala comparabile)", color='#D53F8C')
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(short_names, fontsize=8.5)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Separabilità Spazio Latente')
    axes[1].set_title('Confronto Distance Gap e Discriminabilità d\'')
    axes[1].legend(fontsize=8.5)
    axes[1].grid(axis='x', alpha=0.3)

    # Pannello 3: Confronto cls_roc_auc e cls_pr_auc
    axes[2].barh(y_pos - 0.18, clean_table['cls_roc_auc'], height=0.35, label='Classif. ROC-AUC', color='#2B6CB0')
    axes[2].barh(y_pos + 0.18, clean_table['cls_pr_auc'], height=0.35, label='Classif. PR-AUC', color='#319795')
    axes[2].set_yticks(y_pos)
    axes[2].set_yticklabels(short_names, fontsize=8.5)
    axes[2].invert_yaxis()
    axes[2].set_xlim(0.65, 0.82)
    axes[2].set_xlabel('Score (Test Set)')
    axes[2].set_title('Classificazione Binaria: ROC-AUC & PR-AUC')
    axes[2].legend(fontsize=8.5)
    axes[2].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / 'final_thesis_master_dashboard.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Nessun file *_final_metrics.csv trovato in artifacts/.')